# LeetCode 885 — Spiral Matrix III bằng Python

Link đề: https://leetcode.com/problems/spiral-matrix-iii/ · Độ khó: **Medium** · Chủ đề: Array, Matrix, Simulation

Skeleton LeetCode:

```python
class Solution:
    def spiralMatrixIII(self, rows: int, cols: int, rStart: int, cStart: int) -> List[List[int]]:
        ...
```

Ghi chú: đáp án duy nhất (một danh sách toạ độ **có thứ tự**). Điểm cốt lõi: đường xoắn ốc đi theo các **đoạn** có độ dài `1, 1, 2, 2, 3, 3, …` với hướng quay vòng **Đông → Nam → Tây → Bắc**; ta đi cả ra ngoài lưới nhưng chỉ ghi lại ô **nằm trong** lưới. Oracle: **quy tắc bàn tay phải** (đi thẳng, rẽ phải khi ô bên phải chưa đi qua) — một cách mô tả hoàn toàn khác với dãy độ dài đoạn.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: xuất phát tại `(rStart, cStart)` hướng Đông, đi xoắn ốc theo chiều kim đồng hồ (kể cả ra ngoài lưới) cho tới khi thăm đủ `rows × cols` ô; trả về thứ tự các ô trong lưới.
2. Nắm công cụ Python: mảng hướng `[(0,1),(1,0),(0,-1),(-1,0)]`, `itertools.cycle`/`count`/`chain`, kiểm tra biên `0 <= r < rows`, **cắt đoạn** (clipping) bằng số học, sắp xếp theo khoá.
3. Đi từ **mô phỏng từng bước (độ dài đoạn `1,1,2,2,…`)** → **dùng `itertools` sinh dãy đoạn** → **cắt đoạn trong `O(1)` mỗi đoạn** → **công thức đóng cho chỉ số xoắn ốc rồi sắp xếp**.
4. Hiểu mấu chốt: độ dài đoạn tăng thêm `1` **sau mỗi hai** lần rẽ; vì tia xoắn ốc có thể ra ngoài lưới rồi quay lại, **không dừng** khi ra ngoài mà dừng khi đã thu đủ `rows × cols` ô.
5. Tránh lỗi kinh điển: độ dài đoạn `1,2,3,4` (sai), quay ngược chiều kim đồng hồ, quên ô xuất phát, dừng ngay khi ra ngoài lưới, ghi cả ô ngoài lưới, và đổi chỗ `[r, c]` thành `[c, r]`.

## 2. Đề bài & ví dụ

Bạn đứng ở ô `(rStart, cStart)` của lưới `rows × cols`, quay mặt về hướng **Đông**. Góc Tây-Bắc là hàng và cột đầu tiên, góc Đông-Nam là hàng và cột cuối.

Đi theo hình **xoắn ốc theo chiều kim đồng hồ** để thăm mọi ô. Khi đi ra ngoài biên, tiếp tục đi ngoài lưới (có thể quay lại biên sau đó). Cuối cùng sẽ qua đủ `rows * cols` ô.

Trả về mảng toạ độ các ô của lưới theo **thứ tự đã thăm**.

**Ví dụ 1:** `rows = 1, cols = 4, rStart = 0, cStart = 0` → `[[0,0],[0,1],[0,2],[0,3]]`.

**Ví dụ 2:** `rows = 5, cols = 6, rStart = 1, cStart = 4` → `[[1,4],[1,5],[2,5],[2,4],[2,3],[1,3],[0,3],[0,4],[0,5],[3,5],[3,4],[3,3],[3,2],[2,2],[1,2],[0,2],[4,5],[4,4],[4,3],[4,2],[4,1],[3,1],[2,1],[1,1],[0,1],[4,0],[3,0],[2,0],[1,0],[0,0]]`.

**Ràng buộc:** `1 <= rows, cols <= 100`; `0 <= rStart < rows`; `0 <= cStart < cols`.

In [1]:
import itertools
import random
import time
from typing import List

ex1 = ((1, 4, 0, 0), [[0, 0], [0, 1], [0, 2], [0, 3]])
ex2 = ((5, 6, 1, 4), [[1, 4], [1, 5], [2, 5], [2, 4], [2, 3], [1, 3], [0, 3], [0, 4], [0, 5], [3, 5], [3, 4], [3, 3], [3, 2], [2, 2],
                      [1, 2], [0, 2], [4, 5], [4, 4], [4, 3], [4, 2], [4, 1], [3, 1], [2, 1], [1, 1], [0, 1], [4, 0], [3, 0], [2, 0], [1, 0], [0, 0]])
examples = [ex1, ex2]
for args, want in examples:
    print(args, "->", len(want), "ô, ô đầu", want[0], ", ô cuối", want[-1])

(1, 4, 0, 0) -> 4 ô, ô đầu [0, 0] , ô cuối [0, 3]
(5, 6, 1, 4) -> 30 ô, ô đầu [1, 4] , ô cuối [0, 0]


## 3. Kiến thức nền

### 3.1 Bốn hướng theo chiều kim đồng hồ

Hướng Đông `(0, +1)`, Nam `(+1, 0)`, Tây `(0, −1)`, Bắc `(−1, 0)` (hàng tăng khi đi xuống). Rẽ phải (kim đồng hồ) = chuyển sang hướng kế tiếp trong danh sách, theo chỉ số modulo 4.

In [2]:
DIRS = [(0, 1), (1, 0), (0, -1), (-1, 0)]              # Đông, Nam, Tây, Bắc
names = ["Đông", "Nam", "Tây", "Bắc"]
for i in range(4):
    print(f"{names[i]:4s} {DIRS[i]} -> rẽ phải thành {names[(i + 1) % 4]}")

Đông (0, 1) -> rẽ phải thành Nam
Nam  (1, 0) -> rẽ phải thành Tây
Tây  (0, -1) -> rẽ phải thành Bắc
Bắc  (-1, 0) -> rẽ phải thành Đông


### 3.2 Độ dài các đoạn của xoắn ốc

Từ ô xuất phát, đường xoắn ốc đi các đoạn: Đông `1`, Nam `1`, Tây `2`, Bắc `2`, Đông `3`, Nam `3`, Tây `4`, Bắc `4`, … Độ dài mỗi đoạn là `j // 2 + 1` với `j = 0, 1, 2, …` là số thứ tự đoạn; hướng của đoạn `j` là `DIRS[j % 4]`.

In [3]:
segs = [(names[j % 4], j // 2 + 1) for j in range(10)]
print(segs)
r = c = 0
path = [(r, c)]
for j in range(6):
    dr, dc = DIRS[j % 4]
    for _ in range(j // 2 + 1):
        r, c = r + dr, c + dc
        path.append((r, c))
print(path)

[('Đông', 1), ('Nam', 1), ('Tây', 2), ('Bắc', 2), ('Đông', 3), ('Nam', 3), ('Tây', 4), ('Bắc', 4), ('Đông', 5), ('Nam', 5)]
[(0, 0), (0, 1), (1, 1), (1, 0), (1, -1), (0, -1), (-1, -1), (-1, 0), (-1, 1), (-1, 2), (0, 2), (1, 2), (2, 2)]


### 3.3 Cắt đoạn thẳng với lưới

Đoạn ngang trên hàng `r` chỉ có phần nằm trong lưới khi `0 ≤ r < rows`; khi đó các cột hợp lệ là một **đoạn liên tục** `[0, cols−1]` giao với các cột mà đoạn đi qua. Tính giao bằng `max`/`min` thay vì duyệt từng ô.

### 3.4 `itertools`

`itertools.count(1)` đếm vô hạn, `itertools.cycle(DIRS)` lặp vô hạn, `itertools.chain.from_iterable(repeat(k, 2) for k in count(1))` sinh dãy `1,1,2,2,3,3,…`.

In [4]:
lengths = itertools.chain.from_iterable(itertools.repeat(k, 2) for k in itertools.count(1))
print(list(itertools.islice(lengths, 8)), list(itertools.islice(itertools.cycle("ENWS"), 6)))

[1, 1, 2, 2, 3, 3, 4, 4] ['E', 'N', 'W', 'S', 'E', 'N']


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 2: `5×6`, xuất phát `(1, 4)`.

```
        c: 0  1  2  3  4  5
   r=0     .  .  .  .  .  .
   r=1     .  .  .  .  S  .
   r=2     .  .  .  .  .  .
   r=3     .  .  .  .  .  .
   r=4     .  .  .  .  .  .
```

Các đoạn: Đông 1 → `(1,5)`; Nam 1 → `(2,5)`; Tây 2 → `(2,4)`, `(2,3)`; Bắc 2 → `(1,3)`, `(0,3)`; Đông 3 → `(0,4)`, `(0,5)`, `(0,6)` (ngoài lưới); Nam 3 → `(1,6)`, `(2,6)`, `(3,6)` (ngoài); Tây 4 → `(3,5)`, `(3,4)`, `(3,3)`, `(3,2)` (vào lại lưới)…

Ta ghi lại mọi ô **trong lưới** theo thứ tự, và dừng khi đủ `30` ô.

In [5]:
R, C, r0, c0 = 5, 6, 1, 4
r, c = r0, c0
print("(r, c)    trong lưới?")
steps = [(r, c)]
for j in range(7):
    dr, dc = DIRS[j % 4]
    for _ in range(j // 2 + 1):
        r, c = r + dr, c + dc
        steps.append((r, c))
print(" ".join(f"({a},{b}){'' if 0 <= a < R and 0 <= b < C else '*'}" for a, b in steps), "(* = ngoài lưới)")

(r, c)    trong lưới?
(1,4) (1,5) (2,5) (2,4) (2,3) (1,3) (0,3) (0,4) (0,5) (0,6)* (1,6)* (2,6)* (3,6)* (3,5) (3,4) (3,3) (3,2) (* = ngoài lưới)


## 5. Cách 1 — Mô phỏng từng bước

**Ý tưởng:** đứng tại `(r, c)`; với mỗi đoạn (hướng `DIRS[d]`, độ dài `step`), đi từng bước một: sau mỗi bước, nếu ô nằm trong lưới thì ghi lại. Sau hai đoạn tăng `step` thêm `1`. Dừng khi đã ghi đủ `rows × cols` ô.

In [6]:
def spiral_simulate(rows, cols, rs, cs):
    res = [[rs, cs]]                                     # ô xuất phát luôn nằm trong lưới
    r, c = rs, cs
    d, step = 0, 1
    while len(res) < rows * cols:
        for _ in range(2):                               # mỗi độ dài được dùng cho hai đoạn liên tiếp
            dr, dc = DIRS[d % 4]
            for _ in range(step):
                r, c = r + dr, c + dc
                if 0 <= r < rows and 0 <= c < cols:      # chỉ ghi ô trong lưới
                    res.append([r, c])
            d += 1
        step += 1
    return res


for args, want in examples:
    assert spiral_simulate(*args) == want
print([len(spiral_simulate(*a)) for a, _ in examples])

[4, 30]


In [7]:
# Code nộp LeetCode cho Cách 1
class SolutionSimulate:
    def spiralMatrixIII(self, rows: int, cols: int, rStart: int, cStart: int) -> List[List[int]]:
        dirs = [(0, 1), (1, 0), (0, -1), (-1, 0)]
        res = [[rStart, cStart]]
        r, c, d, step = rStart, cStart, 0, 1
        while len(res) < rows * cols:
            for _ in range(2):
                dr, dc = dirs[d % 4]
                for _ in range(step):
                    r, c = r + dr, c + dc
                    if 0 <= r < rows and 0 <= c < cols:
                        res.append([r, c])
                d += 1
            step += 1
        return res

**Độ phức tạp:** số bước đi tới khi phủ hết lưới là `O(max(rows, cols)²)` (xoắn ốc phải rộng ra cỡ khoảng cách xa nhất tới góc): tối đa `~4·10^4` với `100 × 100`. Bộ nhớ `O(rows·cols)` cho kết quả.

**Nút thắt:** ta bước **từng ô** ngoài lưới. Với một đoạn thẳng nằm hoàn toàn ngoài lưới, có thể bỏ qua trong `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

Hai hướng cải tiến:

1. Gọn hơn: sinh dãy `(hướng, độ dài)` bằng `itertools` (Cách 2).
2. Nhanh hơn: với mỗi đoạn, tính **phần giao** với lưới bằng số học và thêm các ô đó, không bước qua ô ngoài lưới (Cách 3). Hoặc dùng **công thức đóng** cho thứ tự xoắn ốc của mọi ô rồi sắp xếp (Cách 4).

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** `itertools`, **Cách 3** cắt đoạn `O(1)`, **Cách 4** công thức chỉ số + sắp xếp.

### Cách 2 — Sinh dãy đoạn bằng `itertools`

**Giải thích từng đoạn:**

1. `lengths = 1,1,2,2,3,3,…` bằng `chain.from_iterable(repeat(k, 2) for k in count(1))`; `dirs = cycle(DIRS)`.
2. Lặp `zip(lengths, dirs)`: với mỗi `(độ dài, hướng)` đi từng ô, ghi ô trong lưới.
3. Dừng khi đủ `rows × cols`.

In [8]:
def spiral_itertools(rows, cols, rs, cs):
    res = [[rs, cs]]
    r, c = rs, cs
    lengths = itertools.chain.from_iterable(itertools.repeat(k, 2) for k in itertools.count(1))
    for step, (dr, dc) in zip(lengths, itertools.cycle(DIRS)):
        for _ in range(step):
            r, c = r + dr, c + dc
            if 0 <= r < rows and 0 <= c < cols:
                res.append([r, c])
        if len(res) == rows * cols:
            break
    return res


for args, want in examples:
    assert spiral_itertools(*args) == want
print([len(spiral_itertools(*a)) for a, _ in examples])

[4, 30]


In [9]:
# Code nộp LeetCode cho Cách 2
import itertools


class SolutionItertools:
    def spiralMatrixIII(self, rows: int, cols: int, rStart: int, cStart: int) -> List[List[int]]:
        res = [[rStart, cStart]]
        r, c = rStart, cStart
        lengths = itertools.chain.from_iterable(itertools.repeat(k, 2) for k in itertools.count(1))
        for step, (dr, dc) in zip(lengths, itertools.cycle([(0, 1), (1, 0), (0, -1), (-1, 0)])):
            for _ in range(step):
                r, c = r + dr, c + dc
                if 0 <= r < rows and 0 <= c < cols:
                    res.append([r, c])
            if len(res) == rows * cols:
                break
        return res

**Độ phức tạp:** như Cách 1 (`O(max(rows, cols)²)`), nhưng mã ngắn.

### Cách 3 — Cắt đoạn `O(1)` mỗi đoạn

**Ý tưởng:** với đoạn từ `(r, c)` theo hướng `(dr, dc)` dài `L`:

- nếu đoạn **ngang** (`dr = 0`): bỏ qua nếu `r` ngoài `[0, rows−1]`; ngược lại các bước `t = 1..L` có cột `c + dc·t` trong `[0, cols−1]` khi `t` thuộc một khoảng `[t_lo, t_hi]` tính bằng `max`/`min`; thêm các ô đó theo thứ tự `t` tăng;
- đoạn **dọc**: đối xứng.

Sau đó nhảy thẳng tới cuối đoạn `(r + dr·L, c + dc·L)`. Số đoạn là `O(max(rows, cols))` nên tổng chi phí `O(rows·cols + max(rows, cols))`.

In [10]:
def spiral_clip(rows, cols, rs, cs):
    res = [[rs, cs]]
    r, c = rs, cs
    j = 0
    while len(res) < rows * cols:
        dr, dc = DIRS[j % 4]
        L = j // 2 + 1
        if dr == 0:                                       # đoạn ngang: hàng r cố định
            if 0 <= r < rows:
                if dc == 1:
                    lo, hi = max(1, -c), min(L, cols - 1 - c)          # c + t trong [0, cols-1]
                else:
                    lo, hi = max(1, c - (cols - 1)), min(L, c)         # c - t trong [0, cols-1]
                for t in range(lo, hi + 1):
                    res.append([r, c + dc * t])
        else:                                             # đoạn dọc: cột c cố định
            if 0 <= c < cols:
                if dr == 1:
                    lo, hi = max(1, -r), min(L, rows - 1 - r)
                else:
                    lo, hi = max(1, r - (rows - 1)), min(L, r)
                for t in range(lo, hi + 1):
                    res.append([r + dr * t, c])
        r, c = r + dr * L, c + dc * L                     # nhảy tới cuối đoạn
        j += 1
    return res


for args, want in examples:
    assert spiral_clip(*args) == want
print([len(spiral_clip(*a)) for a, _ in examples])

[4, 30]


In [11]:
# Code nộp LeetCode cho Cách 3
class SolutionClip:
    def spiralMatrixIII(self, rows: int, cols: int, rStart: int, cStart: int) -> List[List[int]]:
        dirs = [(0, 1), (1, 0), (0, -1), (-1, 0)]
        res = [[rStart, cStart]]
        r, c, j = rStart, cStart, 0
        while len(res) < rows * cols:
            dr, dc = dirs[j % 4]
            L = j // 2 + 1
            if dr == 0:
                if 0 <= r < rows:
                    if dc == 1:
                        lo, hi = max(1, -c), min(L, cols - 1 - c)
                    else:
                        lo, hi = max(1, c - (cols - 1)), min(L, c)
                    for t in range(lo, hi + 1):
                        res.append([r, c + dc * t])
            else:
                if 0 <= c < cols:
                    if dr == 1:
                        lo, hi = max(1, -r), min(L, rows - 1 - r)
                    else:
                        lo, hi = max(1, r - (rows - 1)), min(L, r)
                    for t in range(lo, hi + 1):
                        res.append([r + dr * t, c])
            r, c = r + dr * L, c + dc * L
            j += 1
        return res

**Độ phức tạp:** `O(rows·cols + max(rows, cols))` thời gian (mỗi đoạn `O(1)` + số ô thêm), `O(rows·cols)` bộ nhớ cho kết quả.

### Cách 4 — Công thức chỉ số xoắn ốc rồi sắp xếp

**Ý tưởng:** tìm công thức đóng `idx(a, b)` = số thứ tự (từ `0`) của ô lệch `(a, b) = (r − rStart, c − cStart)` trong xoắn ốc vô hạn. Rồi sắp xếp mọi ô của lưới theo `idx`.

Gọi `m ≥ 0` (và `m = −1` cho đoạn Đông đầu tiên). Bốn đoạn "vòng `m`" (mỗi đoạn xuất phát tại góc cuối đoạn trước), số thứ tự `n₀ = (2m + 1)²` tại ô `(−m, m + 1)`:

| đoạn | ô (lệch) | điều kiện | `idx` |
|---|---|---|---|
| Nam | `(a, m + 1)` | `b = m + 1`, `−m + 1 ≤ a ≤ m + 1` | `n₀ + (a + m)` |
| Tây | `(m + 1, b)` | `a = m + 1`, `−m − 1 ≤ b ≤ m` | `n₀ + (2m + 1) + (m + 1 − b)` |
| Bắc | `(a, −m − 1)` | `b = −m − 1`, `−m − 1 ≤ a ≤ m` | `n₀ + (2m + 1) + (2m + 2) + (m + 1 − a)` |
| Đông | `(−m − 1, b)` | `a = −m − 1`, `−m ≤ b ≤ m + 2` | `n₀ + (2m + 1) + 2(2m + 2) + (b + m + 1)` |

Ô `(0, 0)` có `idx = 0`. Công thức được **kiểm chứng bằng máy** với mô phỏng (mục 10, 12).

In [12]:
def spiral_index(a, b):
    """Số thứ tự (từ 0) của ô lệch (a, b) = (hàng - rStart, cột - cStart) trong xoắn ốc vô hạn."""
    if a == 0 and b == 0:
        return 0
    # thử từng đoạn: m xác định từ toạ độ cố định của đoạn
    m = b - 1                                            # đoạn Nam: b = m + 1
    if m >= 0 and -m + 1 <= a <= m + 1:
        return (2 * m + 1) ** 2 + (a + m)
    m = a - 1                                            # đoạn Tây: a = m + 1
    if m >= 0 and -m - 1 <= b <= m:
        return (2 * m + 1) ** 2 + (2 * m + 1) + (m + 1 - b)
    m = -b - 1                                           # đoạn Bắc: b = -m - 1
    if m >= 0 and -m - 1 <= a <= m:
        return (2 * m + 1) ** 2 + (2 * m + 1) + (2 * m + 2) + (m + 1 - a)
    m = -a - 1                                           # đoạn Đông: a = -m - 1 (m >= -1: đoạn Đông đầu tiên là m = -1)
    if m >= -1 and -m <= b <= m + 2:
        return (2 * m + 1) ** 2 + (2 * m + 1) + 2 * (2 * m + 2) + (b + m + 1)
    raise AssertionError((a, b))


def spiral_formula(rows, cols, rs, cs):
    cells = [(r, c) for r in range(rows) for c in range(cols)]
    cells.sort(key=lambda rc: spiral_index(rc[0] - rs, rc[1] - cs))
    return [[r, c] for r, c in cells]


for args, want in examples:
    assert spiral_formula(*args) == want
print([len(spiral_formula(*a)) for a, _ in examples], [spiral_index(0, 1), spiral_index(1, 1), spiral_index(1, -1), spiral_index(-1, 2)])

[4, 30] [1, 2, 4, 9]


In [13]:
# Code nộp LeetCode cho Cách 4
class SolutionFormula:
    def spiralMatrixIII(self, rows: int, cols: int, rStart: int, cStart: int) -> List[List[int]]:
        def idx(a, b):
            if a == 0 and b == 0:
                return 0
            m = b - 1
            if m >= 0 and -m + 1 <= a <= m + 1:
                return (2 * m + 1) ** 2 + (a + m)
            m = a - 1
            if m >= 0 and -m - 1 <= b <= m:
                return (2 * m + 1) ** 2 + (2 * m + 1) + (m + 1 - b)
            m = -b - 1
            if m >= 0 and -m - 1 <= a <= m:
                return (2 * m + 1) ** 2 + (2 * m + 1) + (2 * m + 2) + (m + 1 - a)
            m = -a - 1
            return (2 * m + 1) ** 2 + (2 * m + 1) + 2 * (2 * m + 2) + (b + m + 1)

        cells = [[r, c] for r in range(rows) for c in range(cols)]
        cells.sort(key=lambda rc: idx(rc[0] - rStart, rc[1] - cStart))
        return cells

**Độ phức tạp:** `O(rows·cols · log(rows·cols))` thời gian do sắp xếp, `O(rows·cols)` bộ nhớ. Không mô phỏng bước đi nào.

## 8. TRACE từng bước

**Trace Cách 1** với `rows = 5, cols = 6, rStart = 1, cStart = 4` (10 đoạn đầu):

| đoạn `j` | hướng | độ dài | từ | tới | ô ghi vào kết quả |
|---|---|---|---|---|---|
| 0 | Đông | 1 | `(1,4)` | `(1,5)` | `(1,5)` |
| 1 | Nam | 1 | `(1,5)` | `(2,5)` | `(2,5)` |
| 2 | Tây | 2 | `(2,5)` | `(2,3)` | `(2,4)`, `(2,3)` |
| 3 | Bắc | 2 | `(2,3)` | `(0,3)` | `(1,3)`, `(0,3)` |
| 4 | Đông | 3 | `(0,3)` | `(0,6)` | `(0,4)`, `(0,5)` (`(0,6)` ngoài) |
| 5 | Nam | 3 | `(0,6)` | `(3,6)` | không (cột `6` ngoài lưới) |
| 6 | Tây | 4 | `(3,6)` | `(3,2)` | `(3,5)`, `(3,4)`, `(3,3)`, `(3,2)` |

Hàm trace in đúng các dòng này (và dùng Cách 3 để so số ô cắt được):

In [14]:
def trace_segments(rows, cols, rs, cs, max_seg=10):
    r, c = rs, cs
    res = [[rs, cs]]
    print(f"lưới {rows}x{cols}, xuất phát ({rs},{cs}); ô xuất phát được ghi đầu tiên")
    print(f"{'j':>2} {'hướng':<5} {'dài':>3} {'từ':>8} {'tới':>8}  ô ghi vào kết quả")
    for j in range(max_seg):
        dr, dc = DIRS[j % 4]
        L = j // 2 + 1
        added = []
        r0, c0 = r, c
        for _ in range(L):
            r, c = r + dr, c + dc
            if 0 <= r < rows and 0 <= c < cols:
                res.append([r, c])
                added.append((r, c))
        print(f"{j:>2} {names[j % 4]:<5} {L:>3} {str((r0, c0)):>8} {str((r, c)):>8}  {added if added else 'không'}")
        if len(res) == rows * cols:
            break
    print("số ô đã ghi:", len(res), "/", rows * cols)
    return res


part = trace_segments(5, 6, 1, 4)
assert part == ex2[1][:len(part)]

lưới 5x6, xuất phát (1,4); ô xuất phát được ghi đầu tiên
 j hướng dài       từ      tới  ô ghi vào kết quả
 0 Đông    1   (1, 4)   (1, 5)  [(1, 5)]
 1 Nam     1   (1, 5)   (2, 5)  [(2, 5)]
 2 Tây     2   (2, 5)   (2, 3)  [(2, 4), (2, 3)]
 3 Bắc     2   (2, 3)   (0, 3)  [(1, 3), (0, 3)]
 4 Đông    3   (0, 3)   (0, 6)  [(0, 4), (0, 5)]
 5 Nam     3   (0, 6)   (3, 6)  không
 6 Tây     4   (3, 6)   (3, 2)  [(3, 5), (3, 4), (3, 3), (3, 2)]
 7 Bắc     4   (3, 2)  (-1, 2)  [(2, 2), (1, 2), (0, 2)]
 8 Đông    5  (-1, 2)  (-1, 7)  không
 9 Nam     5  (-1, 7)   (4, 7)  không
số ô đã ghi: 16 / 30


**Trace Cách 4:** bảng `idx` của các ô lệch quanh ô xuất phát (hàng `a`, cột `b`): ô `(0,0)` là `0`; ô `(0,1)` là `1` (đoạn Đông đầu tiên); ô `(1,1)` là `2`; ô `(1,0)` là `3`; ô `(1,-1)` là `4`... Sắp xếp các ô của lưới theo `idx` cho đúng thứ tự thăm. Hàm dưới in `idx` của các ô trên hàng `0` (cột `0..3`) và bảng `5×6` quanh gốc:

In [15]:
print("idx của các ô trên hàng 0, cột 0..3 với xuất phát (0,0):", [spiral_index(0, b) for b in range(4)])
print("idx quanh gốc (a từ -2..2, b từ -2..3):")
for a in range(-2, 3):
    print(" ".join(f"{spiral_index(a, b):3d}" for b in range(-2, 4)))

idx của các ô trên hàng 0, cột 0..3 với xuất phát (0,0): [0, 1, 10, 27]
idx quanh gốc (a từ -2..2, b từ -2..3):
 20  21  22  23  24  25
 19   6   7   8   9  26
 18   5   0   1  10  27
 17   4   3   2  11  28
 16  15  14  13  12  29


## 9. Code nộp LeetCode

Chọn **Cách 1** (mô phỏng, đơn giản và an toàn; `~4·10^4` bước với `100 × 100`). Giữ nguyên `class Solution`, `spiralMatrixIII(self, rows, cols, rStart, cStart)`:

In [16]:
class Solution:
    def spiralMatrixIII(self, rows: int, cols: int, rStart: int, cStart: int) -> List[List[int]]:
        dirs = [(0, 1), (1, 0), (0, -1), (-1, 0)]           # Đông, Nam, Tây, Bắc (theo chiều kim đồng hồ)
        res = [[rStart, cStart]]
        r, c, d, step = rStart, cStart, 0, 1
        while len(res) < rows * cols:
            for _ in range(2):                              # mỗi độ dài dùng cho hai đoạn liên tiếp
                dr, dc = dirs[d % 4]
                for _ in range(step):
                    r, c = r + dr, c + dc
                    if 0 <= r < rows and 0 <= c < cols:     # đi ra ngoài lưới vẫn tiếp tục, chỉ ghi ô trong lưới
                        res.append([r, c])
                d += 1
            step += 1
        return res


for args, want in examples:
    assert Solution().spiralMatrixIII(*args) == want
print([len(Solution().spiralMatrixIII(*a)) for a, _ in examples])         # [4, 30]

[4, 30]


## 10. Vì sao đúng

**(a) Mô hình xoắn ốc.** Từ ô xuất phát hướng Đông, đường xoắn ốc theo chiều kim đồng hồ đi các đoạn có độ dài `1, 1, 2, 2, 3, 3, …` (mục 3.2). Đây là **định nghĩa** chuẩn của "xoắn ốc từ giữa ra" và khớp hai ví dụ của đề; ta cũng kiểm chứng nó với **quy tắc bàn tay phải** ở oracle (mục 12A): đi thẳng, rẽ phải ngay khi ô bên phải chưa từng đi qua — một mô tả độc lập cho đúng cùng một đường.

**(b) Đi ra ngoài lưới là cần thiết.** Đường xoắn ốc có thể ra ngoài rồi quay lại (ví dụ 2: `(0,6)` ngoài lưới, sau đó tới `(3,5)` trong lưới). Vì vậy **không dừng** khi ra ngoài; dừng khi đã ghi đủ `rows × cols` ô.

**(c) Hữu hạn.** Xoắn ốc phủ dần mọi ô nguyên của mặt phẳng; sau tối đa `(2·max(rows, cols))²` bước mọi ô của lưới đã được thăm. Vì mỗi ô được thăm **đúng một lần** (xoắn ốc không tự giao), mỗi ô trong lưới được ghi đúng một lần và vòng lặp kết thúc khi đủ `rows × cols`.

**(d) Cách 2** chỉ thay cách sinh dãy đoạn. **Cách 3**: đoạn ngang trên hàng `r` có các ô `(r, c + dc·t)`, `t = 1..L`; ô nằm trong lưới ⇔ `0 ≤ r < rows` và `0 ≤ c + dc·t ≤ cols − 1`. Với `dc = 1`: `t ≥ −c` và `t ≤ cols − 1 − c`; với `dc = −1`: `t ≤ c` và `t ≥ c − (cols − 1)`. Kết hợp với `1 ≤ t ≤ L` cho `[lo, hi]`; thứ tự `t` tăng giữ đúng thứ tự thăm. Đoạn dọc đối xứng.

**(e) Cách 4.** Số thứ tự các ô cuối của mỗi đoạn tính được theo `n₀ = (2m+1)²` (ô `(−m, m+1)` là ô cuối đoạn Đông trước đó và tại `n₀`), cộng độ dài các đoạn Nam `2m+1`, Tây `2m+2`, Bắc `2m+2`, Đông `2m+3`; tổng `(2m+1) + 2(2m+2) + (2m+3) = 8m + 8 = (2m+3)² − (2m+1)²`, nên đoạn Đông kết thúc đúng tại `(2m+3)²`, là `n₀` của vòng kế tiếp. Các điều kiện trong bảng chia mặt phẳng thành các đoạn rời nhau, nên mỗi ô khớp đúng một công thức. Máy kiểm tra: `spiral_index` khớp mô phỏng trên `[−12, 12]²` (mục 12D).

Ta kiểm chứng bằng máy: mọi cách khớp oracle **bàn tay phải** trên **mọi** `rows, cols ≤ 12` và **mọi** ô xuất phát (mục 12D), và khớp nhau trên lưới `100 × 100` (12F).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mô phỏng từng bước | `O(max(R, C)²)` | `O(R·C)` | **đáp án chính**, đơn giản |
| 2. `itertools` | `O(max(R, C)²)` | `O(R·C)` | mã ngắn |
| 3. Cắt đoạn `O(1)` | `O(R·C + max(R, C))` | `O(R·C)` | nhanh nhất |
| 4. Công thức + sắp xếp | `O(R·C log(R·C))` | `O(R·C)` | không mô phỏng |

Đo thực tế (`100 × 100`, xuất phát ở góc — trường hợp xấu nhất cho Cách 1):

In [17]:
def bench(fn, args, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(*args)
        best = min(best, time.perf_counter() - t0)
    return best


for start in [(0, 0), (99, 99), (50, 50)]:
    row = [f"{name} {bench(fn, (100, 100) + start) * 1e3:7.2f} ms" for name, fn in
           [("simulate", spiral_simulate), ("itertools", spiral_itertools), ("clip", spiral_clip), ("formula", spiral_formula)]]
    print(f"100x100 xuất phát {str(start):9s}: " + " | ".join(row))

100x100 xuất phát (0, 0)   : simulate    3.21 ms | itertools    2.85 ms | clip    0.87 ms | formula    3.82 ms
100x100 xuất phát (99, 99) : simulate    2.83 ms | itertools    2.80 ms | clip    0.94 ms | formula    5.44 ms


100x100 xuất phát (50, 50) : simulate    1.35 ms | itertools    1.54 ms | clip    0.92 ms | formula    4.91 ms


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi lưới nhỏ và mọi ô xuất phát với oracle bàn tay phải, (E) ngẫu nhiên có seed + tính chất, (F) giới hạn `100 × 100`.

### A. Ví dụ của đề

In [18]:
def oracle(rows, cols, rs, cs):
    """Oracle độc lập: QUY TẮC BÀN TAY PHẢI. Đi một bước thẳng; sau mỗi bước, nếu ô bên PHẢI (theo hướng mới) chưa từng thăm thì rẽ phải.
    Không dùng dãy độ dài 1,1,2,2,..."""
    visited = {(rs, cs)}
    res = [[rs, cs]]
    r, c, d = rs, cs, 0                                  # hướng Đông
    while len(res) < rows * cols:
        dr, dc = DIRS[d]
        r, c = r + dr, c + dc
        visited.add((r, c))
        if 0 <= r < rows and 0 <= c < cols:
            res.append([r, c])
        rd = (d + 1) % 4                                 # hướng bên phải
        if (r + DIRS[rd][0], c + DIRS[rd][1]) not in visited:
            d = rd
    return res


IMPLS = {
    "simulate": spiral_simulate,
    "itertools": spiral_itertools,
    "clip": spiral_clip,
    "formula": spiral_formula,
    "Solution": lambda *a: Solution().spiralMatrixIII(*a),
    "SolutionSimulate": lambda *a: SolutionSimulate().spiralMatrixIII(*a),
    "SolutionItertools": lambda *a: SolutionItertools().spiralMatrixIII(*a),
    "SolutionClip": lambda *a: SolutionClip().spiralMatrixIII(*a),
    "SolutionFormula": lambda *a: SolutionFormula().spiralMatrixIII(*a),
}


def check(args, want=None, names=None):
    if want is None:
        want = oracle(*args)
    for name in (names or IMPLS):
        got = IMPLS[name](*args)
        assert got == want, (name, args, got[:8], want[:8])


for args, want in examples:
    assert oracle(*args) == want                         # quy tắc bàn tay phải tái tạo đúng ví dụ của đề
    check(args, want)
print("ví dụ đề: oracle bàn tay phải tái tạo đúng cả hai ví dụ; và", len(IMPLS), "cài đặt đúng")

ví dụ đề: oracle bàn tay phải tái tạo đúng cả hai ví dụ; và 9 cài đặt đúng


### B. Edge case

- Lưới `1 × 1`: `[[0, 0]]`.
- Một hàng, một cột (`1 × n`, `n × 1`) với xuất phát ở giữa.
- Xuất phát ở các góc (Tây-Bắc, Đông-Bắc, Tây-Nam, Đông-Nam).
- Xuất phát ở chính giữa lưới vuông.
- Lưới rất dài (`1 × 100`, `100 × 1`).
- Lưới chữ nhật dẹt (`2 × 7`) xuất phát ở một đầu.

In [19]:
cases = [
    (1, 1, 0, 0), (1, 5, 0, 2), (5, 1, 2, 0), (1, 4, 0, 3), (4, 1, 3, 0),
    (3, 3, 0, 0), (3, 3, 0, 2), (3, 3, 2, 0), (3, 3, 2, 2), (3, 3, 1, 1), (4, 4, 1, 1), (4, 4, 2, 2),
    (2, 7, 0, 0), (2, 7, 1, 6), (7, 2, 3, 1), (1, 100, 0, 50), (100, 1, 99, 0), (6, 5, 5, 4),
]
for args in cases:
    check(args)
    got = Solution().spiralMatrixIII(*args)
    assert len(got) == args[0] * args[1] and got[0] == [args[2], args[3]] and len({tuple(x) for x in got}) == len(got)
assert Solution().spiralMatrixIII(1, 1, 0, 0) == [[0, 0]]
assert Solution().spiralMatrixIII(3, 3, 1, 1)[:5] == [[1, 1], [1, 2], [2, 2], [2, 1], [2, 0]]
print(len(cases), "edge case: đúng, đủ ô, ô đầu là ô xuất phát, không trùng")

18 edge case: đúng, đủ ô, ô đầu là ô xuất phát, không trùng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [20]:
def bug_lengths_1234(rows, cols, rs, cs):              # độ dài đoạn 1,2,3,4,... (sai: phải 1,1,2,2,...)
    res = [[rs, cs]]
    r, c, d, step = rs, cs, 0, 1
    while len(res) < rows * cols and step < 6 * max(rows, cols) + 10:      # xoắn ốc sai KHÔNG phủ hết lưới: cần chặn vòng lặp vô hạn
        dr, dc = DIRS[d % 4]
        for _ in range(step):
            r, c = r + dr, c + dc
            if 0 <= r < rows and 0 <= c < cols:
                res.append([r, c])
        d += 1
        step += 1
    return res


def bug_counterclockwise(rows, cols, rs, cs):          # ngược chiều kim đồng hồ: Đông, Bắc, Tây, Nam
    ccw = [(0, 1), (-1, 0), (0, -1), (1, 0)]
    res = [[rs, cs]]
    r, c, d, step = rs, cs, 0, 1
    while len(res) < rows * cols:
        for _ in range(2):
            dr, dc = ccw[d % 4]
            for _ in range(step):
                r, c = r + dr, c + dc
                if 0 <= r < rows and 0 <= c < cols:
                    res.append([r, c])
            d += 1
        step += 1
    return res


def bug_no_start(rows, cols, rs, cs):                  # quên ô xuất phát (nên thiếu một ô: dừng bằng số ô)
    res = []
    r, c, d, step = rs, cs, 0, 1
    while len(res) < rows * cols - 1:
        for _ in range(2):
            dr, dc = DIRS[d % 4]
            for _ in range(step):
                r, c = r + dr, c + dc
                if 0 <= r < rows and 0 <= c < cols:
                    res.append([r, c])
            d += 1
        step += 1
    return res


def bug_stop_outside(rows, cols, rs, cs):              # dừng ngay khi bước ra ngoài lưới
    res = [[rs, cs]]
    r, c, d, step = rs, cs, 0, 1
    while len(res) < rows * cols:
        for _ in range(2):
            dr, dc = DIRS[d % 4]
            for _ in range(step):
                r, c = r + dr, c + dc
                if not (0 <= r < rows and 0 <= c < cols):
                    return res
                res.append([r, c])
            d += 1
        step += 1
    return res


def bug_record_outside(rows, cols, rs, cs):            # ghi cả ô ngoài lưới
    res = [[rs, cs]]
    r, c, d, step = rs, cs, 0, 1
    n_in = 1
    while n_in < rows * cols:
        for _ in range(2):
            dr, dc = DIRS[d % 4]
            for _ in range(step):
                r, c = r + dr, c + dc
                res.append([r, c])
                if 0 <= r < rows and 0 <= c < cols:
                    n_in += 1
            d += 1
        step += 1
    return res


def bug_swapped(rows, cols, rs, cs):                   # trả [c, r] thay vì [r, c]
    return [[c, r] for r, c in spiral_simulate(rows, cols, rs, cs)]


BUGS = [bug_lengths_1234, bug_counterclockwise, bug_no_start, bug_stop_outside, bug_record_outside, bug_swapped]
for bug in BUGS:
    bad = next(((R, C, r, c) for R in range(1, 6) for C in range(1, 6) for r in range(R) for c in range(C)
                if bug(R, C, r, c) != Solution().spiralMatrixIII(R, C, r, c)), None)
    assert bad is not None, bug.__name__
    got = bug(*bad)
    print(f"{bug.__name__:22s} sai trên (rows,cols,r,c) = {bad}: ra {got[:6]}{'...' if len(got) > 6 else ''} ({len(got)} ô), đúng {Solution().spiralMatrixIII(*bad)[:6]}...")

bug_lengths_1234       sai trên (rows,cols,r,c) = (1, 2, 0, 1): ra [[0, 1]] (1 ô), đúng [[0, 1], [0, 0]]...
bug_counterclockwise   sai trên (rows,cols,r,c) = (2, 2, 0, 0): ra [[0, 0], [0, 1], [1, 0], [1, 1]] (4 ô), đúng [[0, 0], [0, 1], [1, 1], [1, 0]]...
bug_no_start           sai trên (rows,cols,r,c) = (1, 1, 0, 0): ra [] (0 ô), đúng [[0, 0]]...
bug_stop_outside       sai trên (rows,cols,r,c) = (1, 2, 0, 1): ra [[0, 1]] (1 ô), đúng [[0, 1], [0, 0]]...
bug_record_outside     sai trên (rows,cols,r,c) = (1, 2, 0, 0): ra [[0, 0], [0, 1], [1, 1]] (3 ô), đúng [[0, 0], [0, 1]]...
bug_swapped            sai trên (rows,cols,r,c) = (1, 2, 0, 0): ra [[0, 0], [1, 0]] (2 ô), đúng [[0, 0], [0, 1]]...


### D. Vét cạn mọi lưới nhỏ và mọi ô xuất phát

Mọi `rows, cols` từ `1` tới `12` và **mọi** ô xuất phát (`Σ rows·cols = 78² = 6 084` bộ). Oracle: quy tắc bàn tay phải. Thêm kiểm chứng công thức `spiral_index` với mô phỏng trên `[−12, 12]²`.

In [21]:
t0 = time.perf_counter()
cnt = 0
for R in range(1, 13):
    for C in range(1, 13):
        for r in range(R):
            for c in range(C):
                want = oracle(R, C, r, c)
                for name in IMPLS:
                    assert IMPLS[name](R, C, r, c) == want, (name, R, C, r, c)
                cnt += 1
assert cnt == 78 * 78
print(cnt, "bộ (rows, cols, rStart, cStart): 9 cài đặt khớp oracle bàn tay phải;", round(time.perf_counter() - t0, 1), "s")

# công thức chỉ số khớp mô phỏng trên [-12, 12]^2
order = {}
r = c = 0
order[(0, 0)] = 0
n = 0
for j in range(60):
    dr, dc = DIRS[j % 4]
    for _ in range(j // 2 + 1):
        r, c = r + dr, c + dc
        n += 1
        order[(r, c)] = n
inside = [(a, b) for a in range(-12, 13) for b in range(-12, 13)]
assert all(spiral_index(a, b) == order[(a, b)] for a, b in inside)
print("spiral_index khớp mô phỏng trên", len(inside), "ô lệch trong [-12, 12]^2")

6084 bộ (rows, cols, rStart, cStart): 9 cài đặt khớp oracle bàn tay phải; 1.7 s
spiral_index khớp mô phỏng trên 625 ô lệch trong [-12, 12]^2


### E. Ngẫu nhiên có seed và tính chất

Lưới ngẫu nhiên tới `40 × 40`, ô xuất phát ngẫu nhiên. Tính chất: (1) kết quả là một **hoán vị** của mọi ô lưới; (2) ô đầu là ô xuất phát; (3) hai ô liên tiếp có **khoảng cách Chebyshev** không vượt `2·max(rows, cols)` (xoắn ốc rộng ra dần) — và với lưới đủ lớn quanh điểm xuất phát (xuất phát cách mọi biên ≥ `k`), `(2k+1)²` ô đầu của xoắn ốc nằm trọn trong lưới.

In [22]:
rng = random.Random(20240885)
for _ in range(500):
    R, C = rng.randint(1, 40), rng.randint(1, 40)
    rs, cs = rng.randrange(R), rng.randrange(C)
    want = oracle(R, C, rs, cs)
    for name in ("simulate", "itertools", "clip", "formula", "Solution"):
        assert IMPLS[name](R, C, rs, cs) == want, (name, R, C, rs, cs)
    got = Solution().spiralMatrixIII(R, C, rs, cs)
    assert sorted(map(tuple, got)) == [(r, c) for r in range(R) for c in range(C)]          # (1) hoán vị mọi ô
    assert got[0] == [rs, cs]                                                              # (2)
    # (3) nếu xuất phát cách mọi biên >= k thì (2k+1)^2 ô đầu của xoắn ốc nằm trong lưới: ô thứ (2k+1)^2 - 1 phải là ô số (2k+1)^2 - 1 của xoắn ốc vô hạn
    k = min(rs, R - 1 - rs, cs, C - 1 - cs)
    if k >= 1:
        first = got[:(2 * k + 1) ** 2]
        assert all(abs(r - rs) <= k and abs(c - cs) <= k for r, c in first)
        assert all(spiral_index(r - rs, c - cs) == i for i, (r, c) in enumerate(first))      # thứ tự đúng bằng chỉ số xoắn ốc
print("500 ca ngẫu nhiên: 5 cài đặt khớp oracle; hoán vị đủ ô, ô đầu đúng, (2k+1)^2 ô đầu nằm trọn khối lân cận")

500 ca ngẫu nhiên: 5 cài đặt khớp oracle; hoán vị đủ ô, ô đầu đúng, (2k+1)^2 ô đầu nằm trọn khối lân cận


### F. Giới hạn `100 × 100`

Một số ô xuất phát điển hình (4 góc, tâm, giữa cạnh, ngẫu nhiên): các cài đặt nhanh phải khớp oracle bàn tay phải; các lưới dẹt `1 × 100`, `100 × 1`.

In [23]:
rng = random.Random(500)
starts = [(0, 0), (0, 99), (99, 0), (99, 99), (50, 50), (0, 50), (50, 0), (99, 50), (50, 99)] + [(rng.randrange(100), rng.randrange(100)) for _ in range(15)]
t0 = time.perf_counter()
for rs, cs in starts:
    want = oracle(100, 100, rs, cs)
    for name in ("simulate", "itertools", "clip", "formula", "Solution", "SolutionSimulate", "SolutionItertools", "SolutionClip", "SolutionFormula"):
        assert IMPLS[name](100, 100, rs, cs) == want, (name, rs, cs)
for args in [(1, 100, 0, 0), (1, 100, 0, 99), (100, 1, 0, 0), (100, 1, 50, 0), (100, 100, 49, 51), (37, 100, 36, 0), (100, 3, 0, 1)]:
    check(args)
print(len(starts) + 7, "lưới lớn: 9 cài đặt khớp oracle bàn tay phải;", round(time.perf_counter() - t0, 1), "s")

31 lưới lớn: 9 cài đặt khớp oracle bàn tay phải; 1.4 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: độ dài đoạn `1, 2, 3, 4, …`

Độ dài chỉ tăng sau **hai** đoạn: `1, 1, 2, 2, 3, 3, …`.

In [24]:
print("ví dụ 2, 8 ô đầu:", bug_lengths_1234(*ex2[0])[:8], "| đúng:", Solution().spiralMatrixIII(*ex2[0])[:8])
assert bug_lengths_1234(*ex2[0]) != ex2[1]

ví dụ 2, 8 ô đầu: [[1, 4], [1, 5], [2, 5], [3, 5], [3, 4], [3, 3], [3, 2], [2, 2]] | đúng: [[1, 4], [1, 5], [2, 5], [2, 4], [2, 3], [1, 3], [0, 3], [0, 4]]


**Vì sao sai:** sau đoạn Đông `1` phải là đoạn Nam `1` (không phải `2`). **Đúng:** `for _ in range(2)` cho mỗi `step`, rồi `step += 1`.

### Lỗi 2: quay ngược chiều kim đồng hồ

Thứ tự hướng phải là Đông → **Nam** → Tây → Bắc (hàng tăng khi đi xuống).

In [25]:
print("ngược kim:", bug_counterclockwise(*ex2[0])[:6], "| đúng:", Solution().spiralMatrixIII(*ex2[0])[:6])
assert bug_counterclockwise(*ex2[0]) != ex2[1]

ngược kim: [[1, 4], [1, 5], [0, 5], [0, 4], [0, 3], [1, 3]] | đúng: [[1, 4], [1, 5], [2, 5], [2, 4], [2, 3], [1, 3]]


**Vì sao sai:** sau Đông phải rẽ **phải** (Nam). **Đúng:** `DIRS = [(0,1),(1,0),(0,-1),(-1,0)]`.

### Lỗi 3: quên ô xuất phát

In [26]:
print("thiếu ô đầu:", bug_no_start(*ex1[0]), "| đúng:", Solution().spiralMatrixIII(*ex1[0]))
assert bug_no_start(*ex1[0]) != ex1[1]

thiếu ô đầu: [[0, 1], [0, 2], [0, 3]] | đúng: [[0, 0], [0, 1], [0, 2], [0, 3]]


**Vì sao sai:** ô xuất phát thuộc lưới và đứng đầu kết quả. **Đúng:** khởi tạo `res = [[rStart, cStart]]`.

### Lỗi 4: dừng ngay khi ra ngoài lưới

In [27]:
print("dừng sớm:", len(bug_stop_outside(*ex2[0])), "ô (cần 30)")
assert len(bug_stop_outside(*ex2[0])) < 30

dừng sớm: 9 ô (cần 30)


**Vì sao sai:** xoắn ốc ra ngoài rồi **quay lại** lưới (ví dụ 2). **Đúng:** tiếp tục đi, dừng khi đủ `rows·cols` ô.

### Lỗi 5: ghi cả ô ngoài lưới

In [28]:
print("ghi ô ngoài:", len(bug_record_outside(*ex2[0])), "ô (cần 30); ô ngoài đầu tiên:", next(p for p in bug_record_outside(*ex2[0]) if not (0 <= p[0] < 5 and 0 <= p[1] < 6)))
assert len(bug_record_outside(*ex2[0])) > 30

ghi ô ngoài: 73 ô (cần 30); ô ngoài đầu tiên: [0, 6]


**Vì sao sai:** đề chỉ yêu cầu ô **trong** lưới. **Đúng:** kiểm tra `0 <= r < rows and 0 <= c < cols` trước khi ghi.

### Lỗi 6 (bonus): đổi chỗ `[c, r]`

In [29]:
print("[c, r]:", bug_swapped(*ex2[0])[:4], "| đúng:", Solution().spiralMatrixIII(*ex2[0])[:4])
assert bug_swapped(*ex2[0]) != ex2[1]

[c, r]: [[4, 1], [5, 1], [5, 2], [4, 2]] | đúng: [[1, 4], [1, 5], [2, 5], [2, 4]]


**Vì sao sai:** toạ độ là `[hàng, cột]`. **Đúng:** `res.append([r, c])`.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "duyệt xoắn ốc / theo hướng quay vòng" → **mảng hướng** `DIRS` cộng chỉ số `d % 4`; xoắn ốc từ **giữa ra** có độ dài đoạn tăng đều (`1,1,2,2,…`); xoắn ốc từ **biên vào** (Spiral Matrix I/II) dùng biên thu hẹp.

**Mẹo:**

1. Dùng `DIRS` theo chiều kim đồng hồ và `d += 1` khi rẽ.
2. Cho phép đi ngoài lưới, chỉ **lọc** khi ghi.
3. Điều kiện dừng theo **số ô đã ghi**, không theo vị trí.
4. Đoạn lặp hai lần cho mỗi độ dài: `for _ in range(2)`.
5. Khi cần nhanh hơn: cắt đoạn bằng `max/min` hoặc công thức chỉ số.
6. Oracle độc lập: quy tắc bàn tay phải.

**Họ bài cùng pattern:** Spiral Matrix (54), Spiral Matrix II (59), Spiral Matrix IV (2326), Walking Robot Simulation (874), Robot Bounded In Circle (1041), Rotate Image (48).

## 15. Biến thể

1. **Xoắn ốc ngược chiều kim đồng hồ** hoặc **bắt đầu hướng khác** (Bắc/Tây/Nam).
2. **Chỉ số thứ tự của ô `(r, c)`** trong xoắn ốc (không cần sinh cả danh sách).
3. **Ô thứ `k` của xoắn ốc** (nghịch đảo của chỉ số).
4. **Spiral Matrix II (59):** điền `1..n²` theo xoắn ốc từ biên vào.
5. **Số ô trong lưới sau `s` bước** (đếm).

Cài biến thể 1 (hướng bắt đầu và chiều quay tuỳ ý, đối chiếu quy tắc bàn tay phải/trái), 2 và 3:

In [30]:
def spiral_generic(rows, cols, rs, cs, start_dir=0, clockwise=True):
    """Xoắn ốc tổng quát: start_dir chỉ số trong DIRS (0 = Đông, 1 = Nam, 2 = Tây, 3 = Bắc); clockwise = rẽ theo kim đồng hồ."""
    res = [[rs, cs]]
    r, c, d, step = rs, cs, start_dir, 1
    turn = 1 if clockwise else -1
    while len(res) < rows * cols:
        for _ in range(2):
            dr, dc = DIRS[d % 4]
            for _ in range(step):
                r, c = r + dr, c + dc
                if 0 <= r < rows and 0 <= c < cols:
                    res.append([r, c])
            d += turn
        step += 1
    return res


def oracle_generic(rows, cols, rs, cs, start_dir, clockwise):
    """Quy tắc bàn tay phải (kim đồng hồ) hoặc bàn tay trái (ngược kim)."""
    visited = {(rs, cs)}
    res = [[rs, cs]]
    r, c, d = rs, cs, start_dir
    turn = 1 if clockwise else -1
    while len(res) < rows * cols:
        r, c = r + DIRS[d % 4][0], c + DIRS[d % 4][1]
        visited.add((r, c))
        if 0 <= r < rows and 0 <= c < cols:
            res.append([r, c])
        nd = (d + turn) % 4
        if (r + DIRS[nd][0], c + DIRS[nd][1]) not in visited:
            d = nd
    return res


for R in range(1, 6):
    for C in range(1, 6):
        for r in range(R):
            for c in range(C):
                for sd in range(4):
                    for cw in (True, False):
                        assert spiral_generic(R, C, r, c, sd, cw) == oracle_generic(R, C, r, c, sd, cw), (R, C, r, c, sd, cw)
assert spiral_generic(5, 6, 1, 4) == ex2[1]
print("biến thể 1: hướng bắt đầu bất kỳ, chiều quay bất kỳ khớp quy tắc bàn tay phải/trái trên mọi lưới <= 5x5")


def kth_cell(k):
    """Biến thể 3: ô lệch (a, b) có số thứ tự k trong xoắn ốc vô hạn (nghịch đảo của spiral_index)."""
    if k == 0:
        return (0, 0)
    m = 0
    while (2 * m + 3) ** 2 <= k:                         # tìm m sao cho (2m+1)^2 <= k < (2m+3)^2
        m += 1
    return _kth_from_t(m, k - (2 * m + 1) ** 2)          # t = số bước kể từ ô (-m, m+1) có số thứ tự (2m+1)^2


def _kth_from_t(m, t):
    if t == 0:
        return (-m, m + 1)
    if t <= 2 * m + 1:
        return (-m + t, m + 1)                           # Nam
    t -= 2 * m + 1
    if t <= 2 * m + 2:
        return (m + 1, m + 1 - t)                        # Tây
    t -= 2 * m + 2
    if t <= 2 * m + 2:
        return (m + 1 - t, -m - 1)                       # Bắc
    t -= 2 * m + 2
    return (-m - 1, -m - 1 + t)                          # Đông


# kiểm chứng nghịch đảo với spiral_index trên 0..(2*12+1)^2 - 1
for k in range(0, (2 * 12 + 1) ** 2):
    a, b = kth_cell(k)
    assert spiral_index(a, b) == k, (k, (a, b))
print("biến thể 2, 3: spiral_index và kth_cell là nghịch đảo của nhau trên", (2 * 12 + 1) ** 2, "ô đầu")

biến thể 1: hướng bắt đầu bất kỳ, chiều quay bất kỳ khớp quy tắc bàn tay phải/trái trên mọi lưới <= 5x5
biến thể 2, 3: spiral_index và kth_cell là nghịch đảo của nhau trên 625 ô đầu


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng | đi từng bước, lọc ô trong lưới | `O(max(R, C)²)` | `O(R·C)` | **đáp án chính** |
| 2. `itertools` | sinh dãy `(hướng, độ dài)` | `O(max(R, C)²)` | `O(R·C)` | ngắn gọn |
| 3. Cắt đoạn | giao đoạn với lưới bằng `max/min` | `O(R·C + max(R, C))` | `O(R·C)` | nhanh nhất |
| 4. Công thức + sắp xếp | `idx(a, b)` đóng | `O(R·C log(R·C))` | `O(R·C)` | không mô phỏng |

Cả bốn khớp oracle bàn tay phải trên mọi lưới `≤ 12 × 12` với mọi ô xuất phát (mục 12D) và trên lưới `100 × 100`.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [31]:
print(Solution().spiralMatrixIII(2, 2, 0, 0), Solution().spiralMatrixIII(1, 3, 0, 1), Solution().spiralMatrixIII(3, 1, 2, 0))

[[0, 0], [0, 1], [1, 1], [1, 0]] [[0, 1], [0, 2], [0, 0]] [[2, 0], [1, 0], [0, 0]]


### Bài 2 (tính tay)

`rows = 3, cols = 3, rStart = 1, cStart = 1` (ô giữa). (a) Liệt kê 9 ô theo thứ tự. (b) Ô nào bị ghi cuối cùng? (c) Có ô ngoài lưới nào bị đi qua trước khi ghi đủ không? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện mô phỏng:

In [32]:
def simulate_todo(rows, cols, rs, cs):
    res = [[rs, cs]]
    r, c, d, step = rs, cs, 0, 1
    while len(res) < rows * cols:
        for _ in range(2):
            dr, dc = DIRS[d % 4]
            for _ in range(step):
                pass    # TODO 1: đi một bước; nếu trong lưới thì ghi [r, c]
            d += 1
        pass            # TODO 2: tăng độ dài đoạn
    return res


# Khi làm xong, bỏ comment để kiểm tra:
# assert simulate_todo(5, 6, 1, 4) == ex2[1] and simulate_todo(1, 4, 0, 0) == ex1[1]

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **`itertools`** (`cycle`, `count`, `chain`, `repeat`):

In [33]:
class MySolution:
    def spiralMatrixIII(self, rows, cols, rStart, cStart):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().spiralMatrixIII(*a) == w for a, w in examples)

### Bài 5 (mini challenge): chỉ số thứ tự của một ô

Viết `visit_order(rows, cols, rs, cs, r, c)` trả **vị trí** (từ 0) của ô `(r, c)` trong kết quả, **không** sinh toàn bộ danh sách: đếm bao nhiêu ô trong lưới có `spiral_index` nhỏ hơn ô đó (hoặc tìm `spiral_index` rồi đếm). Kiểm tra với `Solution().spiralMatrixIII(...).index([r, c])`.

In [34]:
def visit_order(rows, cols, rs, cs, r, c):
    pass            # TODO


# Khi làm xong:
# assert visit_order(5, 6, 1, 4, 0, 0) == 29

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: các đoạn Đông 1, Nam 1, Tây 2, Bắc 2, Đông 3 ...
- Mức 2: `2×2` từ `(0,0)`: Đông `(0,1)`, Nam `(1,1)`, Tây `(1,0)` (trong lưới), xong.
- Mức 3: `[[0,0],[0,1],[1,1],[1,0]]`; `1×3` từ `(0,1)` cho `[[0,1],[0,2],[0,0]]` (sau khi ra ngoài lưới, tia quay lại ô `(0,0)`); `3×1` từ `(2,0)` cho `[[2,0],[1,0],[0,0]]`.

**Bài 2**

- Mức 1: xuất phát `(1,1)`: Đông `(1,2)`, Nam `(2,2)`, Tây `(2,1)`, `(2,0)`.
- Mức 2: Bắc `(1,0)`, `(0,0)`; Đông `(0,1)`, `(0,2)`, `(0,3)`.
- Mức 3: thứ tự `(1,1),(1,2),(2,2),(2,1),(2,0),(1,0),(0,0),(0,1),(0,2)`; ô cuối là `(0,2)`. Đến lúc đủ `9` ô, tia chưa đi ra ngoài lưới ô nào (ô ngoài đầu tiên `(0,3)` đến sau `(0,2)`).

**Bài 3**

- Mức 1: `r, c = r + dr, c + dc`.
- Mức 2: kiểm tra biên rồi `res.append([r, c])`.
- Mức 3: `step += 1` sau vòng `for _ in range(2)`.

**Bài 4**

- Mức 1: `lengths` = `chain.from_iterable(repeat(k, 2) for k in count(1))`.
- Mức 2: `zip(lengths, cycle(DIRS))`.
- Mức 3: dừng khi đủ ô.

**Bài 5**

- Mức 1: `spiral_index(r - rs, c - cs)` là chỉ số trong xoắn ốc vô hạn.
- Mức 2: vị trí trong kết quả = số ô trong lưới có chỉ số nhỏ hơn.
- Mức 3: đếm bằng hai vòng lặp trên lưới.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [35]:
assert Solution().spiralMatrixIII(2, 2, 0, 0) == [[0, 0], [0, 1], [1, 1], [1, 0]]
assert Solution().spiralMatrixIII(1, 3, 0, 1) == [[0, 1], [0, 2], [0, 0]]
assert Solution().spiralMatrixIII(3, 1, 2, 0) == oracle(3, 1, 2, 0)
print("Bài 1 OK:", Solution().spiralMatrixIII(2, 2, 0, 0), Solution().spiralMatrixIII(1, 3, 0, 1), Solution().spiralMatrixIII(3, 1, 2, 0))

Bài 1 OK: [[0, 0], [0, 1], [1, 1], [1, 0]] [[0, 1], [0, 2], [0, 0]] [[2, 0], [1, 0], [0, 0]]


### Bài 2

(a) `(1,1),(1,2),(2,2),(2,1),(2,0),(1,0),(0,0),(0,1),(0,2)`. (b) `(0,2)` cuối cùng. (c) Không có ô ngoài lưới nào được đi qua trước khi ghi đủ (ô cuối `(0,2)` đạt tại đoạn Đông thứ tư).

In [36]:
got = Solution().spiralMatrixIII(3, 3, 1, 1)
assert got == [[1, 1], [1, 2], [2, 2], [2, 1], [2, 0], [1, 0], [0, 0], [0, 1], [0, 2]] == oracle(3, 3, 1, 1)
print("Bài 2 OK: ô cuối", got[-1])

Bài 2 OK: ô cuối [0, 2]


### Bài 3

In [37]:
def simulate_answer(rows, cols, rs, cs):
    res = [[rs, cs]]
    r, c, d, step = rs, cs, 0, 1
    while len(res) < rows * cols:
        for _ in range(2):
            dr, dc = DIRS[d % 4]
            for _ in range(step):
                r, c = r + dr, c + dc
                if 0 <= r < rows and 0 <= c < cols:
                    res.append([r, c])
            d += 1
        step += 1
    return res


assert simulate_answer(5, 6, 1, 4) == ex2[1] and simulate_answer(1, 4, 0, 0) == ex1[1]
for R in range(1, 7):
    for C in range(1, 7):
        assert simulate_answer(R, C, R // 2, C // 2) == oracle(R, C, R // 2, C // 2)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [38]:
class MySolutionAnswer:
    def spiralMatrixIII(self, rows, cols, rStart, cStart):
        res = [[rStart, cStart]]
        r, c = rStart, cStart
        lengths = itertools.chain.from_iterable(itertools.repeat(k, 2) for k in itertools.count(1))
        for step, (dr, dc) in zip(lengths, itertools.cycle(DIRS)):
            if len(res) == rows * cols:
                break
            for _ in range(step):
                r, c = r + dr, c + dc
                if 0 <= r < rows and 0 <= c < cols:
                    res.append([r, c])
        return res


assert all(MySolutionAnswer().spiralMatrixIII(*a) == w for a, w in examples)
for R in range(1, 8):
    for C in range(1, 8):
        for r in range(R):
            for c in range(C):
                assert MySolutionAnswer().spiralMatrixIII(R, C, r, c) == oracle(R, C, r, c)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [39]:
def visit_order_answer(rows, cols, rs, cs, r, c):
    target = spiral_index(r - rs, c - cs)
    return sum(1 for rr in range(rows) for cc in range(cols) if spiral_index(rr - rs, cc - cs) < target)


assert visit_order_answer(5, 6, 1, 4, 0, 0) == 29
rng = random.Random(5)
for _ in range(300):
    R, C = rng.randint(1, 12), rng.randint(1, 12)
    rs, cs = rng.randrange(R), rng.randrange(C)
    r, c = rng.randrange(R), rng.randrange(C)
    assert visit_order_answer(R, C, rs, cs, r, c) == Solution().spiralMatrixIII(R, C, rs, cs).index([r, c])
print("Bài 5 OK: chỉ số thứ tự khớp .index() trên kết quả đầy đủ")

Bài 5 OK: chỉ số thứ tự khớp .index() trên kết quả đầy đủ


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (xoắn ốc từ giữa ra: đoạn `1,1,2,2,3,3,…`, ghi ô trong lưới):**

```python
class Solution:
    def spiralMatrixIII(self, rows, cols, rStart, cStart):
        dirs = [(0, 1), (1, 0), (0, -1), (-1, 0)]      # Đông, Nam, Tây, Bắc
        res = [[rStart, cStart]]
        r, c, d, step = rStart, cStart, 0, 1
        while len(res) < rows * cols:
            for _ in range(2):                         # mỗi độ dài dùng cho hai đoạn
                dr, dc = dirs[d % 4]
                for _ in range(step):
                    r, c = r + dr, c + dc
                    if 0 <= r < rows and 0 <= c < cols:
                        res.append([r, c])
                d += 1
            step += 1
        return res
```

**Checklist trước khi submit:**

- [ ] Hướng theo chiều kim đồng hồ: Đông → Nam → Tây → Bắc (hàng tăng khi đi xuống)?
- [ ] Độ dài đoạn `1, 1, 2, 2, 3, 3, …` (tăng sau **hai** đoạn)?
- [ ] Ô xuất phát nằm đầu kết quả?
- [ ] Đi **ngoài** lưới vẫn tiếp tục, chỉ ghi ô trong lưới; dừng khi đủ `rows·cols` ô?
- [ ] Ghi `[r, c]` (hàng trước, cột sau)?
- [ ] Đã thử `1×4 từ (0,0)`, `5×6 từ (1,4)`, `1×1`, xuất phát ở góc và ở giữa?
- [ ] Đúng tên `class Solution`, `spiralMatrixIII(self, rows, cols, rStart, cStart)`, không in debug.